In [1]:
"""
================================================================================
NOTEBOOK 5 — FedAvg + EF + TopK  |  DenseNet-121  |  T4x2  (Updated)
================================================================================
TopK + Error Feedback — EF recovers accuracy lost by TopK alone.

CHANGES FROM ORIGINAL:
  • Model       : torchvision EfficientNet-B0 → timm DenseNet-121
  • DataParallel: REMOVED — causes CUBLAS crash with frozen BN on GPU 1
  • evaluate()  : unwraps DataParallel defensively
  • Batch       : 96 eval / 24 client

OUTPUT: /kaggle/working/results/results_ef_topk.pkl
================================================================================
"""

import os, random, json, pickle, time, gc
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

try:
    from torch.amp import autocast, GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    _AMP_NEW = False

import torchvision.transforms as transforms

try:
    import timm
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9.0"])
    import timm

from typing import List
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             precision_score, recall_score, confusion_matrix)
from tqdm.auto import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

print("=" * 70)
print("📓 NB5 — FedAvg+EF+TopK  |  DenseNet-121  |  T4x2")
print("=" * 70)
print(f"PyTorch: {torch.__version__} | timm: {timm.__version__}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class Config:
    UNIFIED_CSV: str = "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv"
    STATS_JSON:  str = "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/dataset_stats.json"
    SAVE_DIR:    str = "/kaggle/working/results"

    TRAIN_RATIO: float = 0.70
    VAL_RATIO:   float = 0.15
    TEST_RATIO:  float = 0.15

    MODEL_NAME: str   = "densenet121"
    TIMM_NAME:  str   = "densenet121"
    PRETRAINED: bool  = True
    DROPOUT:    float = 0.4

    BATCH_SIZE:   int   = 96
    CLIENT_BATCH: int   = 24
    FL_ROUNDS:    int   = 25
    LOCAL_EPOCHS: int   = 1

    BACKBONE_LR:  float = 3e-5
    HEAD_LR:      float = 3e-4
    WEIGHT_DECAY: float = 1e-4
    GRAD_CLIP:    float = 1.0

    FL_FREEZE_ROUNDS: int  = 3
    FREEZE_BN_IN_FL:  bool = True

    POS_WEIGHT: float = 2.0
    TOPK_RATIO: float = 0.3
    EF_ON_CPU:  bool  = True

    SAVE_CHECKPOINTS: bool = True
    CHECKPOINT_EVERY: int  = 5

    NUM_CLIENTS:   int   = 5
    NON_IID_ALPHA: float = 0.5

    SEEDS: List[int] = field(default_factory=lambda: [42, 123])

    DEVICE:      str  = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS: int  = 2
    PIN_MEMORY:  bool = True
    USE_AMP:     bool = True


config = Config()

for _sp in [config.STATS_JSON,
            "/kaggle/input/unified-dr-dataset/dataset_stats.json",
            "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/dataset_stats.json"]:
    if os.path.exists(_sp): config.STATS_JSON = _sp; break

for _cp in [config.UNIFIED_CSV,
            "/kaggle/input/unified-dr-dataset/unified_dataset.csv",
            "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv"]:
    if os.path.exists(_cp): config.UNIFIED_CSV = _cp; print(f"✅ CSV: {_cp}"); break

Path(config.SAVE_DIR).mkdir(parents=True, exist_ok=True)
print(f"  TopK ratio : {config.TOPK_RATIO} | EF: ON (CPU)")
print(f"  Batch      : {config.BATCH_SIZE} eval / {config.CLIENT_BATCH} client")


# =============================================================================
# UTILITIES
# =============================================================================

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def worker_init_fn(wid): np.random.seed(torch.initial_seed() % 2**32 + wid)

def interpolate_history(hl, em):
    r  = list(hl); ei = [i for i in range(len(r)) if em[i]]
    if len(ei) <= 1: return r
    for k in range(len(ei)-1):
        i_s, i_e = ei[k], ei[k+1]; v_s, v_e, sp = r[i_s], r[i_e], i_e-i_s
        for j in range(1, sp): r[i_s+j] = v_s + (j/sp)*(v_e-v_s)
    return r


# =============================================================================
# DATASET
# =============================================================================

def get_transforms(split='train'):
    try:
        with open(config.STATS_JSON) as f: st = json.load(f)
        mean, std = st['mean'], st['std']
    except Exception: mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
    norm = transforms.Normalize(mean, std)
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((224,224)), transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(), transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.1),
            transforms.ToTensor(), norm,
        ])
    return transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), norm])

class DRDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
            if self.transform: img = self.transform(img)
            return img, torch.tensor(float(label), dtype=torch.float32)
        except Exception: return torch.zeros(3,224,224), torch.tensor(0.0)

def load_and_split_data(csv_path, seed=42):
    """
    FIXED — matches NB9 EXACTLY:
    • random_state=42 ALWAYS (not seed-dependent)
    • True 70/15/15 two-step stratified split
    • Path remapping for stale Kaggle dataset paths
    The seed parameter is accepted for API compatibility but IGNORED for splitting.
    """
    print(f"\n📂 Loading: {csv_path}")
    df = pd.read_csv(csv_path)

    # ── Path remapping (same logic as NB9) ───────────────────────────────────
    sample  = df['image_path'].iloc[:20].tolist()
    n_valid = sum(1 for p in sample if os.path.exists(p))
    if n_valid < 18:
        print(f"  ⚠️  Broken paths ({n_valid}/20) — scanning /kaggle/input/ ...")
        filename_to_real = {}
        if os.path.exists('/kaggle/input'):
            for slug in sorted(os.listdir('/kaggle/input')):
                for root, _, files in os.walk(f'/kaggle/input/{slug}'):
                    for fname in files:
                        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                            if fname not in filename_to_real:
                                filename_to_real[fname] = os.path.join(root, fname)
        df = df.copy()
        df['image_path'] = df['image_path'].apply(
            lambda p: filename_to_real.get(os.path.basename(p), p)
                      if not os.path.exists(p) else p)
        n_fixed = sum(1 for p in df['image_path'].iloc[:20] if os.path.exists(p))
        print(f"  ✅ After remap: {n_fixed}/20 valid")

    # ── STEP 1: 70% train / 30% temp  — ALWAYS random_state=42 ─────────────
    # CRITICAL: must NOT use seed here — NB9 hardcodes 42 so every seed
    # trains/tests on exactly the same rows, making seed-to-seed variance
    # measure algorithm stochasticity rather than data sampling variation.
    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - config.TRAIN_RATIO),       # 0.30
        stratify=df['binary_label'],
        random_state=42,                            # ← HARDCODED, not seed
    )

    # ── STEP 2: Split temp 50/50 → 15% val / 15% test ───────────────────────
    val_ratio_adj = config.VAL_RATIO / (config.VAL_RATIO + config.TEST_RATIO)  # 0.5
    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - val_ratio_adj),             # 0.5
        stratify=temp_df['binary_label'],
        random_state=42,                            # ← HARDCODED, not seed
    )

    # Convert DataFrame rows to (path, label) tuples
    train_samples = [(row['image_path'], int(row['binary_label']))
                     for _, row in train_df.iterrows()]
    val_samples   = [(row['image_path'], int(row['binary_label']))
                     for _, row in val_df.iterrows()]
    test_samples  = [(row['image_path'], int(row['binary_label']))
                     for _, row in test_df.iterrows()]

    tr_pos = sum(s[1] for s in train_samples)
    print(f"  Train: {len(train_samples):,} (DR+: {tr_pos} = "
          f"{tr_pos/len(train_samples)*100:.1f}%)")
    print(f"  Val:   {len(val_samples):,}")
    print(f"  Test:  {len(test_samples):,}")

    return train_samples, val_samples, test_samples


def create_non_iid_clients(train_samples, num_clients=5, alpha=0.5, seed=42):
    """
    FIXED — matches NB9 EXACTLY:
    • Uses np.random.default_rng (NOT legacy np.random.seed)
    • Per-client val split is 10% (NB9) not 20% (old ablations)
    • Returns {cid: (train_ds, val_ds)} dict directly — same as NB9
    """
    rng = np.random.default_rng(seed)             # ← MATCHES NB9 (not legacy RNG)
    labels      = np.array([s[1] for s in train_samples])
    num_classes = len(np.unique(labels))
    client_indices = [[] for _ in range(num_clients)]

    for c in range(num_classes):
        class_idx = np.where(labels == c)[0]
        rng.shuffle(class_idx)                     # ← default_rng shuffle
        proportions = rng.dirichlet(alpha * np.ones(num_clients))
        proportions = (proportions * len(class_idx)).astype(int)
        proportions[-1] = len(class_idx) - proportions[:-1].sum()
        start = 0
        for cid, count in enumerate(proportions):
            client_indices[cid].extend(class_idx[start:start+count].tolist())
            start += count

    print(f"\n  Non-IID clients (α={alpha}):")
    client_datasets = {}
    for cid in range(num_clients):
        cs = [train_samples[i] for i in client_indices[cid]]
        cl = [s[1] for s in cs]
        pos = sum(cl)
        print(f"    C{cid}: {len(cs):,} | DR+: {pos} ({pos/max(len(cs),1)*100:.1f}%)")

        if len(cs) > 20:
            mc = min(sum(1 for l in cl if l == 0), sum(1 for l in cl if l == 1))
            if mc >= 2:
                c_tr, c_va = train_test_split(
                    cs, test_size=0.1,             # ← 10% like NB9 (was 20%)
                    stratify=[s[1] for s in cs],
                    random_state=seed + cid,
                )
            else:
                c_tr, c_va = train_test_split(
                    cs, test_size=0.1,
                    random_state=seed + cid,
                )
        else:
            c_tr, c_va = cs, cs[:0]

        client_datasets[cid] = (
            DRDataset(c_tr, get_transforms('train')),
            DRDataset(c_va, get_transforms('val')),
        )

    return client_datasets


class DRClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(config.TIMM_NAME, pretrained=pretrained,
                                          num_classes=0, global_pool='avg')
        self.backbone.eval()
        with torch.no_grad():
            fd = int(self.backbone(torch.zeros(1,3,224,224)).shape[-1])
        print(f"  🔧 {config.MODEL_NAME}: feature_dim={fd}")
        self.classifier = nn.Sequential(nn.Linear(fd,512), nn.ReLU(True),
                                        nn.Dropout(dropout), nn.Linear(512,1))
        print(f"  ✅ Trainable: {sum(p.numel() for p in self.parameters() if p.requires_grad):,}")

    def forward(self, x): return self.classifier(self.backbone(x)).squeeze(-1)
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
    def freeze_bn_for_fl(self):
        for m in self.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                m.eval()
                for p in m.parameters(): p.requires_grad = False
    def get_param_groups(self, bb_lr, head_lr):
        return [{'params': self.backbone.parameters(), 'lr': bb_lr},
                {'params': self.classifier.parameters(), 'lr': head_lr}]

def create_model():         return DRClassifier(config.PRETRAINED, config.DROPOUT)
def create_client_model(d): return DRClassifier(False, config.DROPOUT).to(d)


# =============================================================================
# EVALUATION
# =============================================================================

def evaluate(model, loader, device, return_arrays=True):
    m = model.module if isinstance(model, nn.DataParallel) else model
    m.eval(); yt, yp = [], []
    with torch.no_grad():
        for x, y in loader:
            probs = torch.sigmoid(m(x.to(device, non_blocking=True))).cpu().numpy()
            yp.extend(probs); yt.extend(y.numpy())
    yt = np.array(yt); yp = np.array(yp); yh = (yp > 0.5).astype(int)
    try: auc = roc_auc_score(yt, yp)
    except: auc = 0.5
    mets = {'auc': auc, 'f1': f1_score(yt, yh, zero_division=0),
            'precision': precision_score(yt, yh, zero_division=0),
            'recall': recall_score(yt, yh, zero_division=0),
            'accuracy': accuracy_score(yt, yh)}
    if return_arrays: mets.update({'y_true': yt, 'y_prob': yp, 'y_pred': yh})
    return mets


# =============================================================================
# SPARSE COMPRESSION WITH ERROR FEEDBACK
# =============================================================================

class SparsePayload:
    def __init__(self, indices, values, shape, dtype):
        self.indices = indices.astype(np.int32); self.values = values.astype(np.float32)
        self.shape = shape; self.dtype = dtype
    def to_dense(self, device='cpu'):
        d = torch.zeros(self.shape, device=device, dtype=torch.float32).view(-1)
        d[torch.from_numpy(self.indices).long().to(device)] = torch.from_numpy(self.values).float().to(device)
        return d.view(self.shape)
    def get_bytes(self): return self.indices.nbytes + self.values.nbytes + 32


def topk_compress_with_ef(deltas, error_buf, ratio=0.3):
    compressed = {}; new_ef = {}; orig_b = 0; comp_b = 0; ef_norm_total = 0.0

    for name, delta in deltas.items():
        if delta.dtype not in [torch.float32, torch.float16, torch.bfloat16]:
            compressed[name] = delta; b = delta.numel()*delta.element_size()
            orig_b += b; comp_b += b; new_ef[name] = torch.zeros_like(delta, device='cpu')
            continue
        orig_b += delta.numel() * delta.element_size()

        d_cpu = delta.float().cpu()
        d_eff = d_cpu + error_buf[name].float() if name in error_buf else d_cpu.clone()

        flat = d_eff.view(-1); k = max(1, int(len(flat)*ratio))
        _, idx = torch.topk(flat.abs(), k); vals = flat[idx]

        compressed[name] = SparsePayload(idx.numpy(), vals.numpy(), delta.shape, delta.dtype)
        comp_b += compressed[name].get_bytes()

        sent = torch.zeros_like(flat); sent[idx] = vals
        new_error = flat - sent
        new_ef[name] = new_error.view(delta.shape)
        ef_norm_total += new_error.norm().item() ** 2

    return compressed, new_ef, orig_b, comp_b, ef_norm_total**0.5


def sparse_aggregate(global_model, updates, device):
    base  = global_model.module if isinstance(global_model, nn.DataParallel) else global_model
    state = base.state_dict(); total = sum(u['n'] for u in updates)
    for key in state:
        if state[key].dtype in [torch.long, torch.int64]: continue
        acc = torch.zeros_like(state[key], dtype=torch.float32)
        for u in updates:
            w = u['n']/total; sp = u['deltas'][key]
            if isinstance(sp, SparsePayload): acc += w * sp.to_dense(device)
            elif torch.is_tensor(sp):         acc += w * sp.to(device).float()
        state[key] = state[key] + acc.to(state[key].dtype)
    base.load_state_dict(state)


# =============================================================================
# LOCAL TRAINING (FedAvg)
# =============================================================================

def local_train(client_model, train_loader, device, rnd):
    init = {k: v.clone() for k, v in client_model.state_dict().items()}
    client_model.train()
    if config.FREEZE_BN_IN_FL: client_model.freeze_bn_for_fl()
    if rnd < config.FL_FREEZE_ROUNDS: client_model.freeze_backbone()
    else: client_model.unfreeze_backbone()

    optimizer = optim.AdamW(client_model.get_param_groups(config.BACKBONE_LR, config.HEAD_LR),
                            weight_decay=config.WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([config.POS_WEIGHT], device=device))
    if config.USE_AMP: scaler = GradScaler('cuda') if _AMP_NEW else GradScaler()

    for _ in range(config.LOCAL_EPOCHS):
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if config.USE_AMP:
                ctx = autocast('cuda') if _AMP_NEW else autocast()
                with ctx: loss = criterion(client_model(x), y)
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), config.GRAD_CLIP)
                scaler.step(optimizer); scaler.update()
            else:
                criterion(client_model(x), y).backward()
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), config.GRAD_CLIP)
                optimizer.step()

    final = client_model.state_dict()
    return {k: final[k]-init[k] for k in final}, len(train_loader.dataset)


# =============================================================================
# FEDERATED TRAINING LOOP
# =============================================================================

def train_federated(client_datasets, val_loader, test_loader, seed=42):
    set_seed(seed); device = config.DEVICE; t0 = time.time()

    # DataParallel REMOVED — causes CUBLAS crash with frozen BN on GPU 1
    global_model = create_model().to(device)
    print(f"\n  🚀 FedAvg+EF+TopK [{config.MODEL_NAME}] seed={seed} | ratio={config.TOPK_RATIO} | EF=ON")

    history = {'val_auc': [], 'test_auc': [], 'comm_mb': [],
               'comm_mb_cumulative': [], 'orig_mb_cumulative': [],
               'compression_ratio': [], 'ef_norm': []}
    eval_mask  = []; total_comm = 0; total_orig = 0
    last_val   = 0.5; last_test = 0.5
    ef_buffers = {cid: {} for cid in client_datasets}
    eval_rounds = {0, 5, 10, 15, 20, 24}

    if config.SAVE_CHECKPOINTS:
        ckpt = Path(config.SAVE_DIR) / "checkpoints"; ckpt.mkdir(parents=True, exist_ok=True)

    pbar = tqdm(range(config.FL_ROUNDS), desc=f"  EF+TopK seed={seed}")

    for rnd in pbar:
        updates = []; round_orig = 0; round_comp = 0; round_ef_norms = []
        global_state = global_model.state_dict()

        for cid, (train_ds, _) in client_datasets.items():
            dl = DataLoader(train_ds, batch_size=config.CLIENT_BATCH, shuffle=True,
                            num_workers=0, generator=torch.Generator().manual_seed(seed+rnd+cid))
            cm = create_client_model(device); cm.load_state_dict(global_state)
            deltas, n = local_train(cm, dl, device, rnd)
            compressed, new_ef, o_b, c_b, ef_norm = topk_compress_with_ef(
                deltas, ef_buffers[cid], config.TOPK_RATIO)
            ef_buffers[cid] = new_ef; round_ef_norms.append(ef_norm)
            round_orig += o_b; round_comp += c_b
            updates.append({'deltas': compressed, 'n': n})
            del cm, deltas; torch.cuda.empty_cache()

        sparse_aggregate(global_model, updates, device)
        total_orig += round_orig; total_comm += round_comp
        del updates; torch.cuda.empty_cache()

        is_eval = (rnd in eval_rounds)
        if is_eval:
            val_m  = evaluate(global_model, val_loader,  device, return_arrays=False)
            test_m = evaluate(global_model, test_loader, device, return_arrays=False)
            last_val, last_test = val_m['auc'], test_m['auc']

        cr = round_comp/round_orig if round_orig > 0 else 1.0
        history['val_auc'].append(last_val)
        history['test_auc'].append(last_test)
        history['comm_mb'].append(round_comp/1e6)
        history['comm_mb_cumulative'].append(total_comm/1e6)
        history['orig_mb_cumulative'].append(total_orig/1e6)
        history['compression_ratio'].append(cr)
        history['ef_norm'].append(float(np.mean(round_ef_norms)))
        eval_mask.append(is_eval)

        pbar.set_postfix({'val': f"{last_val:.4f}", 'test': f"{last_test:.4f}",
                          'comp': f"{total_comm/1e6:.0f}MB", 'cr': f"{cr:.3f}",
                          'ef': f"{np.mean(round_ef_norms):.2f}",
                          '✓' if is_eval else '·': ''})

        if config.SAVE_CHECKPOINTS and (rnd+1) % config.CHECKPOINT_EVERY == 0:
            torch.save({'round': rnd+1, 'model': global_model.state_dict(), 'history': history},
                       ckpt / f"ef_topk_rnd{rnd+1}_s{seed}.pt")

    for key in ['val_auc', 'test_auc']:
        history[key] = interpolate_history(history[key], eval_mask)

    final = evaluate(global_model, test_loader, device, return_arrays=True)
    elapsed = time.time() - t0
    reduction_pct = (1 - total_comm/total_orig)*100 if total_orig > 0 else 0
    print(f"\n  ✅ seed={seed} | AUC={final['auc']:.4f} | F1={final['f1']:.4f} "
          f"| Comm={total_comm/1e6:.1f}MB ↓{reduction_pct:.1f}% | Time={elapsed/60:.1f}min")

    del global_model; torch.cuda.empty_cache(); gc.collect()
    return {'final': final, 'history': history,
            'total_comm_mb': total_comm/1e6, 'total_orig_mb': total_orig/1e6,
            'reduction_pct': reduction_pct}


# =============================================================================
# MAIN
# =============================================================================

def main():
    print(f"\n{'='*70}")
    print("🚀 NB5 — FedAvg+EF+TopK | DenseNet-121")
    print(f"{'='*70}")

    exp_results = []; total_t0 = time.time()

    for si, seed in enumerate(config.SEEDS):
        print(f"\n  Seed {seed}  ({si+1}/{len(config.SEEDS)})")
        train_s, val_s, test_s = load_and_split_data(config.UNIFIED_CSV, seed)

        val_loader  = DataLoader(DRDataset(val_s,  get_transforms('val')),
                                 batch_size=config.BATCH_SIZE, shuffle=False,
                                 num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)
        test_loader = DataLoader(DRDataset(test_s, get_transforms('val')),
                                 batch_size=config.BATCH_SIZE, shuffle=False,
                                 num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)

        # create_non_iid_clients returns {cid: (train_ds, val_ds)} directly
        # (fixed: uses default_rng + 10% client val split, matching NB9)
        client_ds = create_non_iid_clients(
            train_s, config.NUM_CLIENTS, config.NON_IID_ALPHA, seed)

        res = train_federated(client_ds, val_loader, test_loader, seed=seed)
        res['final'].update({'history': res['history'], 'total_comm_mb': res['total_comm_mb'],
                             'total_orig_mb': res['total_orig_mb'], 'reduction_pct': res['reduction_pct']})
        exp_results.append(res['final'])

    total_elapsed = time.time() - total_t0
    aucs = [r['auc'] for r in exp_results]; f1s = [r['f1'] for r in exp_results]

    results = {
        'FedAvg+EF+TopK': {
            'mean_auc': np.mean(aucs), 'std_auc': np.std(aucs),
            'mean_comm': np.mean([r['total_comm_mb'] for r in exp_results]),
            'mean_orig': np.mean([r['total_orig_mb'] for r in exp_results]),
            'mean_reduction': np.mean([r['reduction_pct'] for r in exp_results]),
            'all_results': exp_results,
        }
    }

    print(f"\n  AUC : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"  F1  : {np.mean(f1s):.4f}")
    print(f"  Comm: {results['FedAvg+EF+TopK']['mean_comm']:.1f} MB ↓{results['FedAvg+EF+TopK']['mean_reduction']:.1f}%")
    print(f"  Time: {total_elapsed/60:.1f} min")

    pkl_out = f"{config.SAVE_DIR}/results_ef_topk.pkl"
    with open(pkl_out, 'wb') as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"\n💾 Saved → {pkl_out}")
    print(f"\n✅ NB5 DONE | AUC={np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"   EF should recover accuracy vs NB4 (TopK alone).")


if __name__ == "__main__":
    main()

📓 NB5 — FedAvg+EF+TopK  |  DenseNet-121  |  T4x2
PyTorch: 2.9.0+cu126 | timm: 1.0.24
  GPU 0: Tesla T4
  GPU 1: Tesla T4
✅ CSV: /kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv
  TopK ratio : 0.3 | EF: ON (CPU)
  Batch      : 96 eval / 24 client

🚀 NB5 — FedAvg+EF+TopK | DenseNet-121

  Seed 42  (1/2)

📂 Loading: /kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv
  ⚠️  Broken paths (12/20) — scanning /kaggle/input/ ...
  ✅ After remap: 20/20 valid
  Train: 11,656 (DR+: 5895 = 50.6%)
  Val:   2,498
  Test:  2,498

  Non-IID clients (α=0.5):
    C0: 1,368 | DR+: 968 (70.8%)
    C1: 1,668 | DR+: 1525 (91.4%)
    C2: 4,323 | DR+: 1450 (33.5%)
    C3: 3,742 | DR+: 1884 (50.3%)
    C4: 555 | DR+: 68 (12.3%)


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169

  🚀 FedAvg+EF+TopK [densenet121] seed=42 | ratio=0.3 | EF=ON


  EF+TopK seed=42:   0%|          | 0/25 [00:00<?, ?it/s]

  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,1

  EF+TopK seed=123:   0%|          | 0/25 [00:00<?, ?it/s]

  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,169
  🔧 densenet121: feature_dim=1024
  ✅ Trainable: 7,479,1